# Force-field comparison for tau-K18


## Setup

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import MDAnalysis as mda
import mdtraj as md
import numpy as np
import seaborn as sns
from matplotlib.ticker import MultipleLocator
from scipy import interpolate
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

path_ff = "/home/tb/adlouet/DATA/PTM_LIBRARY/post_processed_trajectories_2025/comparing_ff/tauk18/"
path_rdc = "/home/tb/adlouet/DATA/RDC/tauk18/amber03/"

force_fields = [('AMBER03', 'AMBER03', 'amber03_rep_box.gro', 'amber03', 'AMBER03'),
                ('AMBER99SB*-ILDN-Q', 'AMBER99SB-ILDN', 'amber99sb_rep_box.gro', 'amber99', 'AMBER99SB-ILDN'),
                ('CHARMM36m', 'CHARMM36', 'charmm_box.gro', 'charmm', 'CHARMM36')]
n_replicas = 5
md_color, hcg_color, exp_color = '#377eb8', '#1b9e77', '#969696'

fig_dir = 'figures_ff_comparison'
cache_dir = 'ff_cache' 
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)


full_width = 7.0
sns.set_theme(style="ticks", context="paper", font="Liberation Sans", rc={
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'axes.linewidth': 0.8, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'mathtext.default': 'regular', 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def save_figure(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(f'{fig_dir}/{name}.{ext}', dpi=600, bbox_inches=fig.bbox_inches)
    plt.show()

def cached(name, compute):
    path = f'{cache_dir}/{name}.pck'
    if os.path.exists(path):
        with open(path, 'rb') as file:
            return pickle.load(file)
    result = compute()
    with open(path, 'wb') as file:
        pickle.dump(result, file)
    return result

## Figure S2a: radius of gyration distributions

In [ ]:
exp_rg = np.array([0, 17, 18.0, 20, 25, 30, 35, 40, 45, 50, 55, 60])
exp_density = np.array([0, 0.0, 0.0005, 0.01, 0.055, 0.07, 0.06, 0.037, 0.017, 0.007, 0.001, 0])
exp_rg_smooth = np.linspace(17, 60, 50)
exp_density_smooth = interpolate.make_interp_spline(exp_rg, exp_density, k=3)(exp_rg_smooth)

def compute_rg(folder, gro):
    xtcs = [path_ff + f'{folder}/rep_{r}/prod/pbc_nojump_2025.xtc' for r in range(11)]
    reps = [r for r, xtc in enumerate(xtcs) if os.path.exists(xtc)][:n_replicas]
    print(folder, 'replicas:', reps)
    return np.concatenate([md.compute_rg(md.load(xtcs[r], top=path_ff + f'{folder}/rep_{r}/box/{gro}'), masses=None) * 10
                           for r in reps])  # nm to A

fig, axes = plt.subplots(1, 3, figsize=(full_width, 2.8), sharex=True, sharey=True, constrained_layout=True)
for ax, (label, folder, gro, *_) in zip(axes, force_fields):
    rg = cached(f'rg_{folder}', lambda: compute_rg(folder, gro))
    ax.hist(rg, bins=50, density=True, color=md_color, alpha=0.7, linewidth=0, label='MD')
    ax.plot(exp_rg_smooth, exp_density_smooth, color='black', linestyle='dashed', linewidth=1.2, label='Experiment')
    ax.set_title(label)
    ax.set_xlabel(r'$R_G$ ($\AA$)')
    ax.set_xlim(0, 80)
    ax.set_ylim(0, 0.094)
    ax.set_yticks([0, 0.04, 0.08])
    sns.despine(ax=ax)
axes[0].set_ylabel('Probability density')
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=2, frameon=False,
           columnspacing=1.0, handlelength=1.4, fontsize=9.5)
save_figure(fig, 'figure_s2a_rg_distributions')

## Figure S2b,c: residual dipolar couplings

In [ ]:
def scale_factor(calculated, experimental):
    return curve_fit(lambda x, a: a * x, calculated, experimental)[0]

def uniform_average(pales):
    return pales[:, :, -4].mean(axis=0)

exp_rdc = np.genfromtxt(path_rdc + 'script/rdc_exp.txt')
universe = mda.Universe(path_rdc + 'rep_0/pdb/pair0.pdb')
non_proline = universe.select_atoms('not resname PRO').residues.resnums + 243
in_experiment = np.in1d(non_proline, exp_rdc[:, 0])
resids = non_proline[in_experiment]
in_simulation = np.in1d(exp_rdc[:, 0], non_proline)
measured = exp_rdc[in_simulation, 1]

hcg_rdc = uniform_average(np.load(path_rdc + 'script/palesN-H_50k-all.npy')[:, in_experiment])

rdc_mean, rdc_sem = {}, {}
for label, _, _, name, _ in force_fields:
    pales = np.load(f'rdc_all_reps_full/palesN-H_{name}_all_reps.npy')
    rep_ids = np.load(f'rdc_all_reps_full/frames_{name}_all_reps.npy')[:, 0]
    reps = np.unique(rep_ids)[:n_replicas]
    keep = np.isin(rep_ids, reps)
    pales, rep_ids = pales[keep][:, in_experiment], rep_ids[keep]
    rdc_mean[label] = uniform_average(pales)
    rep_means = np.array([uniform_average(pales[rep_ids == r]) for r in reps])
    rdc_sem[label] = rep_means.std(axis=0, ddof=1) / np.sqrt(len(reps))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(full_width, 2.75), sharex=True, sharey=True, constrained_layout=True)
for ax, (label, *_) in zip(axes, force_fields):
    scale = scale_factor(rdc_mean[label], measured)
    calculated = rdc_mean[label] * scale
    r = pearsonr(measured, calculated)[0]
    rmsd = np.sqrt(np.mean((calculated - measured) ** 2))
    print(f'{label}: r={r:.2f}, RMSD={rmsd:.2f} Hz, median SEM={np.median(rdc_sem[label] * scale):.2f} Hz, scale={scale[0]:.2f}')
    ax.plot(measured, calculated, 's', mec=md_color, mfc='none', mew=0.8, ms=3)
    ax.plot([-8, 16], [-8, 16], '--', color='grey', linewidth=0.8)
    ax.text(0.05, 0.93, f'r = {r:.2f}', transform=ax.transAxes, va='top')
    ax.set_title(label)
    ax.set_xlabel(r'$D^{exp}_{HN}$ (Hz)')
    ax.set_xlim(-8, 16)
    ax.set_ylim(-8, 16)
    ax.set_xticks([-5, 0, 5, 10, 15])
    ax.set_yticks([-5, 0, 5, 10, 15])
    ax.set_aspect('equal')
    sns.despine(ax=ax)
axes[0].set_ylabel(r'$D^{MD}_{HN}$ (Hz)')
save_figure(fig, 'figure_s2b_rdc_scatter')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(full_width, 6.6), sharex=True, sharey=True, constrained_layout=True)
for ax, (label, *_) in zip(axes, force_fields):
    scale = scale_factor(rdc_mean[label], measured)
    ax.axhline(0, linestyle='--', color='lightgrey', linewidth=0.8)
    ax.plot(exp_rdc[in_simulation, 0], measured, '-o', color=exp_color, linewidth=0.8, ms=2.5, label='Experiment')
    ax.plot(resids, hcg_rdc * scale_factor(hcg_rdc, measured), 's', mec=hcg_color, mfc='none', mew=0.8, ms=3, label='HCG',
            zorder=9)
    ax.errorbar(resids, rdc_mean[label] * scale, yerr=rdc_sem[label] * scale, fmt='s', mec=md_color, mfc='none', mew=0.8,
                ms=3, elinewidth=0.6, ecolor=md_color, label='MD', zorder=10)
    ax.set_title(label, fontsize=10)
    ax.set_xlim(242, 374)
    ax.set_ylim(-10, 18)
    ax.set_xticks(np.arange(240, 380, 20))
    ax.set_yticks([-10, 0, 10])
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    sns.despine(ax=ax)
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside upper center', ncol=3, frameon=False)
axes[-1].set_xlabel('Residue number')
fig.supylabel(r'$D_{HN}$ (Hz)', fontsize=11)
save_figure(fig, 'figure_s2c_rdc_per_residue')

## SAXS goodness of fit by q region

In [ ]:
# Pepsi-SAXS curves over the first 5 replicas (run_saxs_all_reps.py) against the experimental curve
# SASDV46. Each panel gives the mean squared residual in a q region; the dashed line is 1.
q_regions = [(0, 0.05), (0.05, 0.12), (0.12, 0.20), (0.20, 0.41)]
q_labels = ['0\u20130.05', '0.05\u20130.12', '0.12\u20130.20', '0.20\u20130.40']

def read_saxs(path):
    # q (1/nm to 1/A), I, sigma; points without an uncertainty are left out
    rows = []
    for line in open(path):
        fields = line.split()
        if len(fields) == 3:
            try:
                rows.append([float(x) for x in fields])
            except ValueError:
                pass
    rows = np.array(rows)
    rows[:, 0] /= 10.0
    return rows[rows[:, 2] > 0]

def fit_saxs(i_calc, exp):
    weights = 1.0 / exp[:, 2]
    (c, b), *_ = np.linalg.lstsq(np.column_stack([i_calc * weights, weights]), exp[:, 1] * weights, rcond=None)
    i_fit = c * i_calc + b
    return i_fit, np.sum(((exp[:, 1] - i_fit) / exp[:, 2]) ** 2) / (len(exp) - 2)

exp_saxs = read_saxs('SASDV46.dat')
q_exp = exp_saxs[:, 0]
q_pepsi = np.load('saxs_all_reps_full/q_pepsi.npy')

fig, axes = plt.subplots(1, 3, figsize=(full_width, 3.0), sharey=True, constrained_layout=True)
for ax, (label, *_, name) in zip(axes, force_fields):
    curves = np.load(f'saxs_all_reps_full/saxs_{name}_all_reps.npy')
    rep_ids = np.load(f'saxs_all_reps_full/frames_{name}_all_reps.npy')[:, 0]
    keep = np.isin(rep_ids, np.unique(rep_ids)[:n_replicas])
    i_fit, chi2 = fit_saxs(np.interp(q_exp, q_pepsi, curves[keep].mean(axis=0)), exp_saxs)
    residuals = (exp_saxs[:, 1] - i_fit) / exp_saxs[:, 2]
    region_chi2 = [np.mean(residuals[(q_exp >= low) & (q_exp < high)] ** 2) for low, high in q_regions]
    print(f'{label}: chi2={chi2:.2f}, per region:', np.round(region_chi2, 2))
    bars = ax.bar(q_labels, region_chi2, color=md_color, width=0.7)
    ax.bar_label(bars, fmt='%.1f', padding=2, fontsize=9.5)
    ax.axhline(1, linestyle='--', color='grey', linewidth=0.8)
    ax.text(0.95, 0.93, rf'$\chi^2_r$ = {chi2:.2f}', transform=ax.transAxes, ha='right', va='top')
    ax.set_title(label)
    ax.set_xlabel(r'$q$ region ($\AA^{-1}$)')
    ax.tick_params(axis='x', rotation=45)
    sns.despine(ax=ax)
axes[0].set_ylabel(r'Mean $(\Delta I/\sigma)^2$')
axes[0].set_ylim(0, 24)
save_figure(fig, 'saxs_chi2_by_q_region')